# Algorithm 19 — KLDM-PPR-diffcsppp

This notebook is a fresh debug surface for **Algorithm 19**.

It follows the `kldm_ppr_diffcsppp_codex_tutorial.md` flow:

1. verify the DiffCSP++ symmetry backend step by step,
2. verify the KLDM clean denoiser algebra,
3. verify one PPR project step and one repeated kernel,
4. keep endpoint evaluation on the same KLDMPLUS facit path through `sample_evaluation.evaluate_csp_reconstruction(...)` and `StructureMatcher`.

This notebook intentionally imports **Algorithm 19 only** as the high-level method backend. Shared KLDMPLUS loaders and evaluation helpers are used, but no algorithms 10-18 are imported.


In [12]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print(f'ROOT={ROOT}')


ROOT=/workspace


In [13]:
import os
import time
import math
import random
import traceback
import importlib
from collections import Counter
from dataclasses import dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml

import kldmPlus.algorithm19_kldm_ppr_diffcsppp as algo19_backend
algo19_backend = importlib.reload(algo19_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction
from kldmPlus.symmetry import (
    build_diffcsppp_payload_from_syminfo,
    build_diffcsppp_symmetry_payload,
    build_symmetry_frame_bridge,
    estimate_semantic_transport_for_reference_order,
    oracle_spacegroup_from_case,
    payload_site_ids,
    payload_site_slices,
    project_expanded_frac_to_anchor_space,
    reconstruct_expanded_frac_from_anchor_coords,
    transport_vanilla_frac_to_standardized_frame_with_tau,
)

ALGORITHM19_MODE = algo19_backend.ALGORITHM19_MODE
ALGORITHM19_SHORT_NAME = algo19_backend.ALGORITHM19_SHORT_NAME
ALGORITHM19_DESCRIPTION = algo19_backend.ALGORITHM19_DESCRIPTION
ALGORITHM19_RELATION_TO_PPR = algo19_backend.ALGORITHM19_RELATION_TO_PPR
Algorithm19State = algo19_backend.Algorithm19State
Algorithm19Config = algo19_backend.Algorithm19Config
build_oracle_diffcsppp_payload_from_structure = algo19_backend.build_oracle_diffcsppp_payload_from_structure
c_w_ops = algo19_backend.c_w_ops
c_w_ops_payload_frame = algo19_backend.c_w_ops_payload_frame
center_velocity = algo19_backend.center_velocity
coordinate_score_from_model_output = algo19_backend.coordinate_score_from_model_output
expand_anchors_to_full = algo19_backend.expand_anchors_to_full
graph_mean_norm = algo19_backend.graph_mean_norm
kldm_clean_fractional_denoiser_Df = algo19_backend.kldm_clean_fractional_denoiser_Df
kldm_renoise_from_f0 = algo19_backend.kldm_renoise_from_f0
kldm_ppr_noise_chart = algo19_backend.kldm_ppr_noise_chart
map_payload_to_model_frame = algo19_backend.map_payload_to_model_frame
map_model_to_payload_frame = algo19_backend.map_model_to_payload_frame
payload_expand_identity_rmse = algo19_backend.payload_expand_identity_rmse
ppr_kernel_ops = algo19_backend.ppr_kernel_ops
project_full_to_wyckoff_ops_with_payload = algo19_backend.project_full_to_wyckoff_ops_with_payload
ppr_project_step_ops = algo19_backend.ppr_project_step_ops
project_full_to_anchors = algo19_backend.project_full_to_anchors
project_full_to_wyckoff_ops = algo19_backend.project_full_to_wyckoff_ops
torus_mse = algo19_backend.torus_mse
torus_rmse = algo19_backend.torus_rmse
torus_sin_mse = algo19_backend.torus_sin_mse
wrap01 = algo19_backend.wrap01
wrapdiff = algo19_backend.wrapdiff

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

PROFILE = os.environ.get('KLDM_ALGO19_PROFILE', 'laptop').strip().lower()

def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop

def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]

SAMPLE_SEED = int(profile_default('KLDM_ALGO19_SEED', '2', '2'))
GRAPH_IDS = parse_int_list(profile_default('KLDM_ALGO19_GRAPH_IDS', '2', '2,3,5'))
ALGO19_T_VALUES = tuple(float(v.strip()) for v in str(profile_default('KLDM_ALGO19_T_VALUES', '0.2,0.3', '0.2,0.3')).split(',') if v.strip())
ALGO19_M_VALUES = tuple(int(v.strip()) for v in str(profile_default('KLDM_ALGO19_M_VALUES', '1,2', '1,2')).split(',') if v.strip())
ALGO19_PROJ_STEPS = int(profile_default('KLDM_ALGO19_PROJ_STEPS', '6', '8'))
ALGO19_LR = float(profile_default('KLDM_ALGO19_LR', '2e-2', '2e-2'))
ALGO19_LAMBDA_NOISE = float(profile_default('KLDM_ALGO19_LAMBDA_NOISE', '1e-2', '1e-2'))
ALGO19_GRAD_CLIP = float(profile_default('KLDM_ALGO19_GRAD_CLIP', '10.0', '10.0'))
ALGO19_ANCHOR_MODE = str(profile_default('KLDM_ALGO19_ANCHOR_MODE', 'soft', 'soft'))
ALGO19_DENOISER_VARIANT = str(profile_default('KLDM_ALGO19_DENOISER_VARIANT', 'minus', 'minus'))
ALGO19_COORDINATE_SCORE_MODE = str(profile_default('KLDM_ALGO19_COORDINATE_SCORE_MODE', 'direct', 'direct'))
ALGO19_FRAC_TOL = float(profile_default('KLDM_ALGO19_FRAC_TOL', '1e-3', '1e-3'))
ALGO19_SOFT_ANCHOR_TOL = float(profile_default('KLDM_ALGO19_SOFT_ANCHOR_TOL', '1e-5', '1e-5'))

print(f'profile={PROFILE} graphs={GRAPH_IDS} t_values={ALGO19_T_VALUES} M_values={ALGO19_M_VALUES}')
print(f'proj_steps={ALGO19_PROJ_STEPS} lr={ALGO19_LR} lambda_noise={ALGO19_LAMBDA_NOISE} anchor_mode={ALGO19_ANCHOR_MODE} coordinate_score_mode={ALGO19_COORDINATE_SCORE_MODE}')


profile=laptop graphs=[2] t_values=(0.2, 0.3) M_values=(1, 2)
proj_steps=6 lr=0.02 lambda_noise=0.01 anchor_mode=soft coordinate_score_mode=direct


In [14]:
set_seed(SAMPLE_SEED)
runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()
full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
if not available_graph_ids:
    raise RuntimeError(f'No requested graph ids are present. requested={GRAPH_IDS} available=1..{full_num_graphs}')
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.tolist()
print(f'loaded_graph_ids={available_graph_ids} ptr={ptr}')


mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test


dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
loaded_graph_ids=[2] ptr=[0, 16]


In [15]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int


result_tables: dict[str, pd.DataFrame] = {}
payload_cache: dict[int, Any] = {}


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    return dict(sorted(Counter(arr).items()))


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = (
        (edge_index[0] >= start)
        & (edge_index[0] < end)
        & (edge_index[1] >= start)
        & (edge_index[1] < end)
    )
    return (edge_index[:, mask] - start).detach().clone()


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
        'num_atoms': int(end - start),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            composition=composition_counter(g['h']),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out


GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    return SimpleNamespace(pos=pos, batch=batch_index, num_atoms=num_atoms, edge_node_index=edge_node_index)


def build_case_structure(case: GraphCase):
    return build_structure_from_sample(
        case.gt_frac,
        case.gt_l_feature,
        case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
    )


def build_oracle_payload(case: GraphCase):
    if case.graph_id in payload_cache:
        return payload_cache[case.graph_id]
    structure = build_case_structure(case)
    payload = build_oracle_diffcsppp_payload_from_structure(
        standardized_structure=structure,
        requested_spacegroup=oracle_spacegroup_from_case(case),
        tol=1e-2,
    )
    bridge = build_symmetry_frame_bridge(
        vanilla_structure=structure,
        standardization='refined',
        symprec=1e-2,
        angle_tolerance=5.0,
    )
    tau_ref, assignment_ref, rmse_ref = estimate_semantic_transport_for_reference_order(
        standardized_reference_frac_coords=np.asarray(payload.expanded_frac_coords, dtype=float),
        standardized_reference_atomic_numbers=np.asarray(payload.expanded_atomic_numbers, dtype=int),
        bridge=bridge,
    )
    linear_std_to_model = np.asarray(bridge.standardized_to_vanilla_linear, dtype=float)
    linear_model_to_std = np.linalg.inv(linear_std_to_model)
    model_to_payload_order = np.asarray(assignment_ref, dtype=int)
    payload = replace(
        payload,
        debug_info={
            **(payload.debug_info or {}),
            'payload_frame': 'pyxtal_refined_standardized',
            'transport_method': bridge.standardized_to_vanilla_method,
            'transport_rmse': float(rmse_ref),
            'model_reference_frac_coords': np.asarray(structure.frac_coords, dtype=float).tolist(),
            'model_to_payload_linear': linear_model_to_std.tolist(),
            'model_to_payload_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'model_to_payload_order': model_to_payload_order.tolist(),
            'payload_to_model_linear': linear_std_to_model.tolist(),
            'payload_to_model_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'payload_to_model_order': np.asarray(assignment_ref, dtype=int).tolist(),
        },
    )
    payload_cache[case.graph_id] = payload
    return payload




def build_wrong_spacegroup_payload(case: GraphCase, payload) -> Any | None:
    letters = list(payload.wyckoff_letters)
    atom_types = np.asarray(payload.anchor_atomic_numbers, dtype=int).tolist()
    for delta in range(1, 33):
        for sign in (-1, 1):
            candidate_sg = int(payload.spacegroup) + sign * delta
            if candidate_sg < 1 or candidate_sg > 230 or candidate_sg == int(payload.spacegroup):
                continue
            try:
                wrong = build_diffcsppp_payload_from_syminfo(
                    spacegroup_number=candidate_sg,
                    wyckoff_letters=letters,
                    atom_types=atom_types,
                )
            except Exception:
                continue
            if int(wrong.num_atoms) != int(payload.num_atoms):
                continue
            return replace(
                wrong,
                anchor_frac_coords=np.asarray(payload.anchor_frac_coords, dtype=float).copy(),
                debug_info={**(payload.debug_info or {}), **(wrong.debug_info or {}), 'source': 'wrong_spacegroup_same_letters'},
            )
    return None

def build_wrong_payload(case: GraphCase):
    payload = build_oracle_payload(case)
    wrong_sg = build_wrong_spacegroup_payload(case, payload)
    if wrong_sg is not None:
        return wrong_sg
    anchor_species = np.asarray(payload.anchor_atomic_numbers, dtype=int)
    if anchor_species.size > 1 and len(set(anchor_species.tolist())) > 1:
        rotated_species = np.roll(anchor_species, 1)
        wrong = build_diffcsppp_payload_from_syminfo(
            spacegroup_number=int(payload.spacegroup),
            wyckoff_letters=list(payload.wyckoff_letters),
            atom_types=rotated_species.tolist(),
        )
        return replace(
            wrong,
            anchor_frac_coords=np.asarray(payload.anchor_frac_coords, dtype=float).copy(),
            debug_info={**(payload.debug_info or {}), **(wrong.debug_info or {}), 'source': 'wrong_species_assignment'},
        )
    return replace(payload, anchor_index=np.roll(np.asarray(payload.anchor_index, dtype=int), 1), debug_info={**(payload.debug_info or {}), 'source': 'wrong_anchor_roll_noncrystallographic'})


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    result = evaluate_csp_reconstruction(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    return {
        'match': bool(result.match),
        'valid': bool(result.valid),
        'rmse': result.rmse,
        'frac_rmse': result.frac_rmse,
        'min_pair_distance': result.min_pair_distance,
        'metric_source': 'sample_evaluation.evaluate_csp_reconstruction / StructureMatcher',
    }


def algo19_times(case: GraphCase, t_value: float):
    dtype = case.gt_frac.dtype
    device = case.gt_frac.device
    t_graph = torch.as_tensor([[float(t_value)]], device=device, dtype=dtype)
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), float(t_value), device=device, dtype=dtype)
    return t_graph, t_nodes


def make_noisy_state(case: GraphCase, *, t_value: float, seed: int = SAMPLE_SEED):
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(round(1000.0 * float(t_value))))
    t_graph, t_nodes = algo19_times(case, float(t_value))
    f_t, v_t, eps_v, eps_r, r_t = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    state = Algorithm19State(
        f=f_t.detach().clone(),
        v=v_t.detach().clone(),
        l=case.gt_l_feature.detach().clone(),
        atom_types=case.atomic_numbers.detach().clone(),
        node_index=batch_view.batch.detach().clone(),
        edge_node_index=batch_view.edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )
    return state, {'eps_v': eps_v.detach().clone(), 'eps_r': eps_r.detach().clone(), 'r_t': r_t.detach().clone()}


def config19(M: int = 1) -> Algorithm19Config:
    return Algorithm19Config(
        M=int(M),
        proj_steps=int(ALGO19_PROJ_STEPS),
        lr=float(ALGO19_LR),
        lambda_noise=float(ALGO19_LAMBDA_NOISE),
        grad_clip=float(ALGO19_GRAD_CLIP),
        anchor_mode=str(ALGO19_ANCHOR_MODE),
        denoiser_variant=str(ALGO19_DENOISER_VARIANT),
        coordinate_score_mode=str(ALGO19_COORDINATE_SCORE_MODE),
        soft_anchor_tol=float(ALGO19_SOFT_ANCHOR_TOL),
    )


def frame_precondition_ok(case: GraphCase, payload, tol: float = 1e-6) -> tuple[bool, float]:
    gt_model_resid = float(c_w_ops(case.gt_frac, payload).item())
    return bool(gt_model_resid < tol), gt_model_resid


def build_payload_with_order_variant(payload, variant: str):
    debug = dict(payload.debug_info or {})
    base_assignment = np.asarray(debug.get('payload_to_model_order', []), dtype=int)
    if base_assignment.size == 0:
        return payload
    if variant == 'assignment':
        model_to_payload_order = base_assignment.copy()
    elif variant == 'inverse_assignment':
        model_to_payload_order = np.argsort(base_assignment)
    else:
        raise ValueError(f'Unknown order variant: {variant}')
    return replace(
        payload,
        debug_info={
            **debug,
            'order_variant': variant,
            'model_to_payload_order': model_to_payload_order.tolist(),
        },
    )


loaded_graphs= [2] sg= [4]


## Baseline vs PPR sampler

The backend/operator issue has moved out of the way, so the notebook now focuses on the real question: can a KLDM predictor-corrector sampler benefit from PPR with the new Wyckoff backend, and under what optimization settings?


In [26]:
import time
from kldmPlus.utils.time import iter_sampling_times


PPR_FULL_STEPS = 1000
PPR_PROJECTION_INTERVAL = 100
PPR_DEFAULT_PROJ_STEPS = 100
PPR_DEFAULT_LAMBDA0 = 10.0
PPR_DEFAULT_LAMBDA_FLOOR = 1e-6
PPR_DEFAULT_M = 3
PPR_DEBUG_T_VALUES = (0.8, 0.5, 0.3)
PPR_DEBUG_LAMBDA0_VALUES = (0.1, 1.0, 10.0)
PPR_DEBUG_M_VALUES = (0, 1)
PPR_DEBUG_GRAPH_IDS = tuple(case.graph_id for case in GRAPH_CASES if int(case.graph_id) in {2, 3, 5}) or tuple(case.graph_id for case in GRAPH_CASES)


def single_graph_batch(case: GraphCase):
    selected = batch.index_select([int(case.graph_idx0)]) if hasattr(batch, 'index_select') else batch[int(case.graph_idx0)]
    if isinstance(selected, list):
        selected = batch.__class__.from_data_list(selected)
    return selected.to(runner.device)


def tdm_sigma_summary(model, *, t_nodes: torch.Tensor, ref_f: torch.Tensor, ref_v: torch.Tensor) -> dict[str, float]:
    tau = model.tdm.T * t_nodes
    sigma_r = model.tdm.match_dims(model.tdm.wrapped_gaussian_sigma_r_t(tau), ref_f)
    sigma_v = model.tdm.match_dims(model.tdm.vel_scale * model.tdm.gaussian_velocity_sigma(tau), ref_v)
    sigma_r_rms = torch.sqrt(sigma_r.square().mean())
    sigma_v_rms = torch.sqrt(sigma_v.square().mean())
    sigma_eff = torch.sqrt(0.5 * (sigma_r_rms.square() + sigma_v_rms.square()))
    return {
        't_mean': float(t_nodes.float().mean().item()),
        'sigma_r_rms': float(sigma_r_rms.item()),
        'sigma_v_rms': float(sigma_v_rms.item()),
        'sigma_eff': float(sigma_eff.item()),
    }


def lambda_schedule_tdm(
    model,
    *,
    t_nodes: torch.Tensor,
    ref_f: torch.Tensor,
    ref_v: torch.Tensor,
    lambda0: float,
    lambda_floor: float = PPR_DEFAULT_LAMBDA_FLOOR,
) -> tuple[torch.Tensor, dict[str, float]]:
    stats = tdm_sigma_summary(model, t_nodes=t_nodes, ref_f=ref_f, ref_v=ref_v)
    t_mean = torch.as_tensor(stats['t_mean'], device=ref_f.device, dtype=ref_f.dtype)
    sigma_eff = torch.as_tensor(stats['sigma_eff'], device=ref_f.device, dtype=ref_f.dtype)
    lambda_eff = float(lambda0) * (t_mean.square() / (4.0 * sigma_eff.square() + 4.0))
    lambda_eff = torch.clamp(lambda_eff, min=float(lambda_floor))
    return lambda_eff, stats


def paired_renoise_from_f0(
    *,
    model,
    f0: torch.Tensor,
    t_nodes: torch.Tensor,
    node_index: torch.Tensor,
    epsilon_v: torch.Tensor,
    epsilon_r: torch.Tensor,
):
    return model.tdm.sample_noisy_state(
        t=t_nodes,
        f0=f0,
        index=node_index,
        epsilon_v=epsilon_v,
        epsilon_r=epsilon_r,
    )


def sample_paired_eps_like(case: GraphCase, *, seed: int, ref: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    set_seed(int(seed) + 20000 * int(case.graph_id))
    eps_v = torch.randn_like(ref)
    eps_r = torch.randn_like(ref)
    batch_index = torch.zeros(ref.shape[0], device=ref.device, dtype=torch.long)
    eps_v = runner.model.tdm.scatter_center(eps_v, index=batch_index) if hasattr(runner.model.tdm, 'scatter_center') else center_velocity(eps_v, batch_index)
    eps_r = runner.model.tdm.scatter_center(eps_r, index=batch_index) if hasattr(runner.model.tdm, 'scatter_center') else center_velocity(eps_r, batch_index)
    return eps_v, eps_r


def make_algo19_state_from_raw(
    *,
    f: torch.Tensor,
    v: torch.Tensor,
    l: torch.Tensor,
    atom_types: torch.Tensor,
    node_index: torch.Tensor,
    edge_node_index: torch.Tensor,
    t_graph: torch.Tensor,
    t_nodes: torch.Tensor,
) -> Algorithm19State:
    return Algorithm19State(
        f=f.detach().clone(),
        v=v.detach().clone(),
        l=l.detach().clone().reshape(-1),
        atom_types=atom_types.detach().clone(),
        node_index=node_index.detach().clone(),
        edge_node_index=edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )


def project_step_scheduled(
    *,
    state: Algorithm19State,
    payload,
    model,
    proj_steps: int,
    lr: float,
    lambda0: float,
    lambda_floor: float,
    grad_clip: float,
    denoiser_variant: str,
    coordinate_score_mode: str,
    lambda_q: float = 1e-6,
):
    symmetry_context = algo19_backend.make_algorithm19_symmetry_context(
        payload,
        device=state.f.device,
        dtype=state.f.dtype,
    )
    clean_current = kldm_clean_fractional_denoiser_Df(
        model=model,
        f=state.f,
        v=state.v,
        l=state.l,
        atom_types=state.atom_types,
        t_graph=state.t_graph,
        t_nodes=state.t_nodes,
        node_index=state.node_index,
        edge_index=state.edge_node_index,
        variant=denoiser_variant,
        coordinate_score_mode=coordinate_score_mode,
    )
    symmetry_context = replace(
        symmetry_context,
        q_ref=algo19_backend.initialize_runtime_q_ref_from_model_frame(
            clean_current,
            payload,
            q_init=symmetry_context.q_ref,
            lambda_q=float(lambda_q),
        ),
    )
    xi_r = torch.zeros_like(state.f, requires_grad=True)
    xi_v = torch.zeros_like(state.v, requires_grad=True)
    optimizer = torch.optim.Adam([xi_r, xi_v], lr=float(lr))
    logs = []
    params = list(model.parameters()) if hasattr(model, 'parameters') else []
    old_requires_grad = [bool(p.requires_grad) for p in params]
    was_training = bool(model.training)
    model.eval()
    for p in params:
        p.requires_grad_(False)

    try:
        for step_idx in range(int(proj_steps)):
            optimizer.zero_grad(set_to_none=True)
            f_var, v_var = kldm_ppr_noise_chart(
                model=model,
                f_t=state.f,
                v_t=state.v,
                xi_r=xi_r,
                xi_v=xi_v,
                t_nodes=state.t_nodes,
                node_index=state.node_index,
            )
            f0_hat = kldm_clean_fractional_denoiser_Df(
                model=model,
                f=f_var,
                v=v_var,
                l=state.l,
                atom_types=state.atom_types,
                t_graph=state.t_graph,
                t_nodes=state.t_nodes,
                node_index=state.node_index,
                edge_index=state.edge_node_index,
                variant=denoiser_variant,
                coordinate_score_mode=coordinate_score_mode,
            )
            z_payload = algo19_backend.map_model_to_payload_reference_chart(f0_hat, payload)
            chart_details = algo19_backend.c_w_dof_chart_payload_frame(
                z_payload,
                payload,
                q_ref=symmetry_context.q_ref,
                lambda_q=float(lambda_q),
            )
            lambda_eff, lambda_stats = lambda_schedule_tdm(
                model,
                t_nodes=state.t_nodes,
                ref_f=state.f,
                ref_v=state.v,
                lambda0=float(lambda0),
                lambda_floor=float(lambda_floor),
            )
            c_value = chart_details['loss']
            prox = xi_r.square().mean() + xi_v.square().mean()
            loss = c_value + lambda_eff * prox
            loss.backward()
            torch.nn.utils.clip_grad_norm_([xi_r, xi_v], max_norm=float(grad_clip))
            optimizer.step()
            logs.append({
                'step': int(step_idx),
                'loss': float(loss.detach().item()),
                'c_value': float(c_value.detach().item()),
                'prox': float(prox.detach().item()),
                'lambda_eff': float(lambda_eff.detach().item()),
                't_mean': float(lambda_stats['t_mean']),
                'sigma_eff': float(lambda_stats['sigma_eff']),
                'xi_r_norm': float(torch.sqrt(xi_r.detach().square().mean()).item()),
                'xi_v_norm': float(torch.sqrt(xi_v.detach().square().mean()).item()),
                'projection_move_model': float(
                    torus_rmse(
                        f0_hat,
                        algo19_backend.map_payload_reference_chart_to_model_frame(chart_details['z_proj'], payload),
                    ).detach().item()
                ),
                'q_step_norm': float(torch.linalg.norm(chart_details['delta_q']).detach().item()),
            })

        with torch.no_grad():
            f_star, v_star = kldm_ppr_noise_chart(
                model=model,
                f_t=state.f,
                v_t=state.v,
                xi_r=xi_r,
                xi_v=xi_v,
                t_nodes=state.t_nodes,
                node_index=state.node_index,
            )
            f0_star = kldm_clean_fractional_denoiser_Df(
                model=model,
                f=f_star,
                v=v_star,
                l=state.l,
                atom_types=state.atom_types,
                t_graph=state.t_graph,
                t_nodes=state.t_nodes,
                node_index=state.node_index,
                edge_index=state.edge_node_index,
                variant=denoiser_variant,
                coordinate_score_mode=coordinate_score_mode,
            )
            z_payload_star = algo19_backend.map_model_to_payload_reference_chart(f0_star, payload)
            final_chart = algo19_backend.c_w_dof_chart_payload_frame(
                z_payload_star,
                payload,
                q_ref=symmetry_context.q_ref,
                lambda_q=float(lambda_q),
            )
    finally:
        for p, req in zip(params, old_requires_grad):
            p.requires_grad_(req)
        if was_training:
            model.train()

    return {
        'f_star': f_star.detach().clone(),
        'v_star': v_star.detach().clone(),
        'f0_star': f0_star.detach().clone(),
        'q_star': final_chart['q_star'].detach().clone(),
        'z_proj_payload': final_chart['z_proj'].detach().clone(),
        'logs': logs,
        'symmetry_context': symmetry_context,
    }


def ppr_kernel_scheduled(
    *,
    state: Algorithm19State,
    payload,
    model,
    M: int,
    proj_steps: int,
    lr: float,
    lambda0: float,
    lambda_floor: float,
    grad_clip: float,
    denoiser_variant: str,
    coordinate_score_mode: str,
    soft_anchor_tol: float,
    local_projection_tol: float,
    lambda_q: float = 1e-6,
    epsilon_sequence: list[tuple[torch.Tensor, torch.Tensor]] | None = None,
):
    symmetry_context = algo19_backend.make_algorithm19_symmetry_context(
        payload,
        device=state.f.device,
        dtype=state.f.dtype,
    )
    current = replace(state)
    repeat_logs = []

    for repeat_idx in range(int(M)):
        project = project_step_scheduled(
            state=current,
            payload=payload,
            model=model,
            proj_steps=int(proj_steps),
            lr=float(lr),
            lambda0=float(lambda0),
            lambda_floor=float(lambda_floor),
            grad_clip=float(grad_clip),
            denoiser_variant=denoiser_variant,
            coordinate_score_mode=coordinate_score_mode,
            lambda_q=float(lambda_q),
        )
        soft_anchor_constraint = float(
            c_w_ops(
                project['f0_star'],
                payload,
                q_ref=symmetry_context.q_ref,
                lambda_q=float(lambda_q),
            ).detach().item()
        )
        projection_move_model = float(
            torus_rmse(
                project['f0_star'],
                algo19_backend.map_payload_reference_chart_to_model_frame(project['z_proj_payload'], payload),
            ).detach().item()
        )
        soft_anchor_feasible = bool(soft_anchor_constraint < float(soft_anchor_tol))
        chart_branch_status = 'local' if projection_move_model < float(local_projection_tol) else 'branch_jump'
        if project['q_star'] is not None and soft_anchor_feasible and chart_branch_status == 'local':
            symmetry_context = replace(symmetry_context, q_ref=project['q_star'].detach().clone())
        f0_anchor = project['f0_star']

        eps_v = eps_r = None
        if epsilon_sequence is not None and repeat_idx < len(epsilon_sequence):
            eps_v, eps_r = epsilon_sequence[repeat_idx]
        f_new, v_new, epsilon_v, epsilon_r, r_t = model.tdm.sample_noisy_state(
            t=current.t_nodes,
            f0=f0_anchor,
            index=current.node_index,
            epsilon_v=eps_v,
            epsilon_r=eps_r,
        )
        current = replace(current, f=f_new.detach().clone(), v=v_new.detach().clone())
        repeat_logs.append({
            'repeat': int(repeat_idx),
            'c_anchor_soft': soft_anchor_constraint,
            'soft_anchor_feasible': soft_anchor_feasible,
            'projection_move_model': projection_move_model,
            'chart_branch_status': chart_branch_status,
            'q_ref_updated': bool(project['q_star'] is not None and soft_anchor_feasible and chart_branch_status == 'local'),
            'optimizer_tail': project['logs'][-1] if project['logs'] else None,
            'epsilon_v_rms': float(torch.sqrt(epsilon_v.square().mean()).detach().item()),
            'epsilon_r_rms': float(torch.sqrt(epsilon_r.square().mean()).detach().item()),
            'r_t_rms': float(torch.sqrt(r_t.square().mean()).detach().item()),
        })

    return {
        'state': current,
        'logs': repeat_logs,
    }


### 1. Sampler comparison

A fair high-level comparison between a plain `PC(1000)` chain and a `PPR-KLDM(1000)` chain that injects projection every 50 steps. The PPR chain uses the same predictor-corrector backbone and only changes the project-renoise kernel.


In [ ]:
def run_pc_sampler_case(case: GraphCase, *, n_steps: int = PPR_FULL_STEPS, tau: float = 0.25, seed: int = SAMPLE_SEED) -> dict[str, Any]:
    set_seed(int(seed) + 10000 * int(case.graph_id) + 17)
    graph_batch = single_graph_batch(case)
    payload = build_oracle_payload(case)
    started = time.perf_counter()
    with torch.no_grad():
        f_t, v_t, l_t, h_t = runner.model.sample_CSP_algorithm4(
            n_steps=10,#int(n_steps),
            batch=graph_batch,
            tau=float(tau),
        )
    elapsed_s = float(time.perf_counter() - started)
    endpoint = evaluate_arrays(case, f_t, l_t.reshape(-1), h_t)
    return {
        'test': 'algo19_sampler_compare',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'method': 'pc_1000',
        'n_steps': int(n_steps),
        'projection_interval': np.nan,
        'M': 0,
        'lambda0': np.nan,
        'runtime_s': elapsed_s,
        'final_c': float(c_w_ops(f_t, payload).item()),
        'frac_rmse': float(endpoint['frac_rmse']),
        'rmse': float(endpoint['rmse']) if endpoint['rmse'] is not None else float('nan'),
        'match': bool(endpoint['match']),
        'valid': bool(endpoint['valid']),
        'PASS': True,
        'status': 'INFO',
    }


def run_ppr_sampler_case(
    case: GraphCase,
    *,
    n_steps: int = PPR_FULL_STEPS,
    projection_interval: int = PPR_PROJECTION_INTERVAL,
    M: int = PPR_DEFAULT_M,
    proj_steps: int = PPR_DEFAULT_PROJ_STEPS,
    lambda0: float = PPR_DEFAULT_LAMBDA0,
    lambda_floor: float = PPR_DEFAULT_LAMBDA_FLOOR,
    tau: float = 0.25,
    seed: int = SAMPLE_SEED,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    set_seed(int(seed) + 10000 * int(case.graph_id) + 17)
    graph_batch = single_graph_batch(case)
    payload = build_oracle_payload(case)
    state = runner.model._prepare_csp_sampling(
        batch=graph_batch,
        n_steps=int(n_steps),
        t_start=1.0,
        t_final=1e-6,
    )

    projection_trace = []
    started = time.perf_counter()
    for step_idx, times in enumerate(iter_sampling_times(batch=state['batch'], grid=state['sampling_time_grid']), start=1):
        with torch.no_grad():
            preds_curr = runner.model.score_network(
                t=times.now.graph,
                pos=state['f_t'],
                v=state['v_t'],
                h=state['a_t'],
                l=state['l_t'],
                node_index=state['node_index'],
                edge_node_index=state['edge_node_index'],
            )
            state['f_t'], state['v_t'] = state['sampling_tdm'].reverse_step_predictor(
                t=times.now.nodes,
                f_t=state['f_t'],
                v_t=state['v_t'],
                pred_v=preds_curr['v'],
                index=state['node_index'],
                dt=times.dt,
            )
            if times.t_next_float >= 1e-3:
                preds_next = runner.model.score_network(
                    t=times.next.graph,
                    pos=state['f_t'],
                    v=state['v_t'],
                    h=state['a_t'],
                    l=state['l_t'],
                    node_index=state['node_index'],
                    edge_node_index=state['edge_node_index'],
                )
                state['f_t'], state['v_t'] = state['sampling_tdm'].reverse_step_corrector(
                    t=times.next.nodes,
                    f_t=state['f_t'],
                    v_t=state['v_t'],
                    pred_v=preds_next['v'],
                    dt=times.dt,
                    index=state['node_index'],
                    tau=float(tau),
                )
                state['l_t'] = runner.model._reverse_lattice_sampling_step(
                    t=times.next.lattice,
                    x_t=state['l_t'],
                    pred=preds_next['l'],
                    dt=times.dt,
                    num_atoms=state['batch'].num_atoms,
                )

        if step_idx % int(projection_interval) != 0:
            continue

        algo_state = make_algo19_state_from_raw(
            f=state['f_t'],
            v=state['v_t'],
            l=state['l_t'],
            atom_types=state['a_t'],
            node_index=state['node_index'],
            edge_node_index=state['edge_node_index'],
            t_graph=times.next.graph,
            t_nodes=times.next.nodes,
        )
        kernel = ppr_kernel_scheduled(
            state=algo_state,
            payload=payload,
            model=runner.model,
            M=int(M),
            proj_steps=int(proj_steps),
            lr=float(ALGO19_LR),
            lambda0=float(lambda0),
            lambda_floor=float(lambda_floor),
            grad_clip=float(ALGO19_GRAD_CLIP),
            denoiser_variant=str(ALGO19_DENOISER_VARIANT),
            coordinate_score_mode=str(ALGO19_COORDINATE_SCORE_MODE),
            soft_anchor_tol=float(ALGO19_SOFT_ANCHOR_TOL),
            local_projection_tol=5e-2,
        )
        state['f_t'] = kernel['state'].f.detach().clone()
        state['v_t'] = kernel['state'].v.detach().clone()

        kernel_tail = kernel['logs'][-1] if kernel['logs'] else {}
        projection_trace.append({
            'test': 'algo19_sampler_projection_trace',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'method': 'ppr_kldm_1000_proj50',
            'step': int(step_idx),
            't_next': float(times.t_next_float),
            'M': int(M),
            'lambda0': float(lambda0),
            'c_anchor_soft': float(kernel_tail.get('c_anchor_soft', float('nan'))),
            'projection_move_model': float(kernel_tail.get('projection_move_model', float('nan'))),
            'soft_anchor_feasible': bool(kernel_tail.get('soft_anchor_feasible', False)),
            'q_ref_updated': bool(kernel_tail.get('q_ref_updated', False)),
            'status': 'INFO',
            'PASS': True,
        })

    elapsed_s = float(time.perf_counter() - started)
    if state['restore_training']:
        state['score_network'].train()

    endpoint = evaluate_arrays(case, state['f_t'], state['l_t'].reshape(-1), state['a_t'])
    feasible_count = sum(bool(row['soft_anchor_feasible']) for row in projection_trace)
    compare_row = {
        'test': 'algo19_sampler_compare',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'method': 'ppr_kldm_1000_proj50',
        'n_steps': int(n_steps),
        'projection_interval': int(projection_interval),
        'M': int(M),
        'lambda0': float(lambda0),
        'runtime_s': elapsed_s,
        'projection_count': len(projection_trace),
        'soft_anchor_feasible_fraction': float(feasible_count / max(len(projection_trace), 1)),
        'final_c': float(c_w_ops(state['f_t'], payload).item()),
        'frac_rmse': float(endpoint['frac_rmse']),
        'rmse': float(endpoint['rmse']) if endpoint['rmse'] is not None else float('nan'),
        'match': bool(endpoint['match']),
        'valid': bool(endpoint['valid']),
        'PASS': True,
        'status': 'INFO',
    }
    return compare_row, projection_trace


sampler_rows = []
projection_trace_rows = []
for case in GRAPH_CASES:
    sampler_rows.append(run_pc_sampler_case(case))
    compare_row, trace_rows = run_ppr_sampler_case(
        case,
        n_steps=500, #PPR_FULL_STEPS,
        projection_interval=PPR_PROJECTION_INTERVAL,
        M=PPR_DEFAULT_M,
        proj_steps=PPR_DEFAULT_PROJ_STEPS,
        lambda0=PPR_DEFAULT_LAMBDA0,
    )
    sampler_rows.append(compare_row)
    projection_trace_rows.extend(trace_rows)

sampler_compare_df = dataframe_result('algo19_sampler_compare', sampler_rows)
sampler_compare_df = sampler_compare_df.sort_values(['graph', 'method']).reset_index(drop=True)
display(sampler_compare_df)

sampler_projection_trace_df = dataframe_result('algo19_sampler_projection_trace', projection_trace_rows)
sampler_projection_trace_df = sampler_projection_trace_df.sort_values(['graph', 'step']).reset_index(drop=True) if not sampler_projection_trace_df.empty else sampler_projection_trace_df
display(sampler_projection_trace_df)


<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>test</th>
      <th>graph</th>
      <th>space_group</th>
      <th>method</th>
      <th>n_steps</th>
      <th>projection_interval</th>
      <th>M</th>
      <th>lambda0</th>
      <th>runtime_s</th>
      <th>final_c</th>
      <th>frac_rmse</th>
      <th>rmse</th>
      <th>match</th>
      <th>valid</th>
      <th>PASS</th>
      <th>status</th>
      <th>projection_count</th>
      <th>soft_anchor_feasible_fraction</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>algo19_sampler_compare</td>
      <td>2</td>
      <td>4</td>
      <td>pc_1000</td>
      <td>1000</td>
      <td>NaN</td>
      <td>0</td>
      <td>NaN</td>
      <td>262.205060</td>
      <td>0.021416</td>
      <td>0.176130</td>
      <td>0.389759</td>
      <td>True</td>
      <td>True</td>
      <td>True</td>
      <td>INFO</td>
      <td>NaN</td>
      <td>NaN</td>
    </tr>
    <tr>
      <th>1</th>
      <td>algo19_sampler_compare</td>
      <td>2</td>
      <td>4</td>
      <td>ppr_kldm_1000_proj50</td>
      <td>100</td>
      <td>50.0</td>
      <td>1</td>
      <td>10.0</td>
      <td>115.489225</td>
      <td>0.034469</td>
      <td>0.171776</td>
      <td>0.478219</td>
      <td>True</td>
      <td>True</td>
      <td>True</td>
      <td>INFO</td>
      <td>2.0</td>
      <td>0.0</td>
    </tr>
  </tbody>
</table>
</div>

## Optimization debug

These tests are now centered on the real failure mode: whether projected anchors survive KLDM renoising. We use a time-dependent proximity weight, paired renoise noise, stronger projection, `M > 1`, and an acceptance/rollback gate for debugging.


In [25]:
def clean_f0_and_c(state: Algorithm19State, payload) -> tuple[torch.Tensor, float]:
    clean = kldm_clean_fractional_denoiser_Df(
        model=runner.model,
        f=state.f,
        v=state.v,
        l=state.l,
        atom_types=state.atom_types,
        t_graph=state.t_graph,
        t_nodes=state.t_nodes,
        node_index=state.node_index,
        edge_index=state.edge_node_index,
        variant=ALGO19_DENOISER_VARIANT,
        coordinate_score_mode=ALGO19_COORDINATE_SCORE_MODE,
    )
    return clean, float(c_w_ops(clean, payload).item())


def fixed_t_paired_debug_row(
    case: GraphCase,
    *,
    t_value: float,
    lambda0: float,
    M: int,
    proj_steps: int = PPR_DEFAULT_PROJ_STEPS,
    seed: int = SAMPLE_SEED,
) -> dict[str, Any]:
    payload = build_oracle_payload(case)
    state, noise_info = make_noisy_state(case, t_value=float(t_value), seed=int(seed))
    clean_before, c_base_clean = clean_f0_and_c(state, payload)
    lambda_eff, lambda_stats = lambda_schedule_tdm(
        runner.model,
        t_nodes=state.t_nodes,
        ref_f=state.f,
        ref_v=state.v,
        lambda0=float(lambda0),
        lambda_floor=PPR_DEFAULT_LAMBDA_FLOOR,
    )

    eps_v = noise_info['eps_v'].detach().clone()
    eps_r = noise_info['eps_r'].detach().clone()
    f_renoise, v_renoise, *_ = paired_renoise_from_f0(
        model=runner.model,
        f0=clean_before,
        t_nodes=state.t_nodes,
        node_index=state.node_index,
        epsilon_v=eps_v,
        epsilon_r=eps_r,
    )
    renoise_state = replace(state, f=f_renoise.detach().clone(), v=v_renoise.detach().clone())
    clean_renoise, c_renoise_redenoise = clean_f0_and_c(renoise_state, payload)

    kernel = ppr_kernel_scheduled(
        state=state,
        payload=payload,
        model=runner.model,
        M=int(M),
        proj_steps=int(proj_steps),
        lr=float(ALGO19_LR),
        lambda0=float(lambda0),
        lambda_floor=float(PPR_DEFAULT_LAMBDA_FLOOR),
        grad_clip=float(ALGO19_GRAD_CLIP),
        denoiser_variant=str(ALGO19_DENOISER_VARIANT),
        coordinate_score_mode=str(ALGO19_COORDINATE_SCORE_MODE),
        soft_anchor_tol=float(ALGO19_SOFT_ANCHOR_TOL),
        local_projection_tol=5e-2,
        epsilon_sequence=[(eps_v, eps_r)] + [None for _ in range(max(int(M) - 1, 0))],
    )
    kernel_tail = kernel['logs'][-1] if kernel['logs'] else {}
    clean_ppr_redenoise, c_ppr_redenoise = clean_f0_and_c(kernel['state'], payload)

    project = project_step_scheduled(
        state=state,
        payload=payload,
        model=runner.model,
        proj_steps=int(proj_steps),
        lr=float(ALGO19_LR),
        lambda0=float(lambda0),
        lambda_floor=float(PPR_DEFAULT_LAMBDA_FLOOR),
        grad_clip=float(ALGO19_GRAD_CLIP),
        denoiser_variant=str(ALGO19_DENOISER_VARIANT),
        coordinate_score_mode=str(ALGO19_COORDINATE_SCORE_MODE),
    )
    optimizer_tail = project['logs'][-1] if project['logs'] else {}
    c_anchor_soft = float(c_w_ops(project['f0_star'], payload).item())
    proposed_better = bool(c_ppr_redenoise <= c_renoise_redenoise + 1e-12)
    local_move = bool(float(kernel_tail.get('projection_move_model', float('inf'))) < 5e-2)
    accepted = bool(proposed_better and local_move)

    return {
        'test': 'algo19_paired_fixed_t_debug',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        't_value': float(t_value),
        'lambda0': float(lambda0),
        'lambda_eff': float(lambda_eff.detach().item()),
        'sigma_eff': float(lambda_stats['sigma_eff']),
        'M': int(M),
        'proj_steps': int(proj_steps),
        'c_base_clean': float(c_base_clean),
        'c_anchor_soft': float(c_anchor_soft),
        'c_renoise_redenoise': float(c_renoise_redenoise),
        'c_ppr_redenoise': float(c_ppr_redenoise),
        'ppr_beats_renoise': bool(c_ppr_redenoise < c_renoise_redenoise + 1e-12),
        'projection_move_model': float(kernel_tail.get('projection_move_model', float('nan'))),
        'soft_anchor_feasible': bool(kernel_tail.get('soft_anchor_feasible', False)),
        'xi_r_norm': float(optimizer_tail.get('xi_r_norm', float('nan'))),
        'xi_v_norm': float(optimizer_tail.get('xi_v_norm', float('nan'))),
        'frac_rmse_renoise_redenoise': float(evaluate_arrays(case, clean_renoise, state.l, case.atomic_numbers)['frac_rmse']),
        'frac_rmse_ppr_redenoise': float(evaluate_arrays(case, clean_ppr_redenoise, state.l, case.atomic_numbers)['frac_rmse']),
        'accept_gate': bool(accepted),
        'rollback_would_trigger': bool(not accepted),
        'PASS': bool(c_ppr_redenoise < c_renoise_redenoise + 1e-12),
        'status': 'PASS' if bool(c_ppr_redenoise < c_renoise_redenoise + 1e-12) else 'WARN',
    }


def optimizer_trace_rows(
    case: GraphCase,
    *,
    t_value: float,
    lambda0: float,
    proj_steps: int = PPR_DEFAULT_PROJ_STEPS,
    seed: int = SAMPLE_SEED,
) -> list[dict[str, Any]]:
    payload = build_oracle_payload(case)
    state, _ = make_noisy_state(case, t_value=float(t_value), seed=int(seed))
    project = project_step_scheduled(
        state=state,
        payload=payload,
        model=runner.model,
        proj_steps=int(proj_steps),
        lr=float(ALGO19_LR),
        lambda0=float(lambda0),
        lambda_floor=float(PPR_DEFAULT_LAMBDA_FLOOR),
        grad_clip=float(ALGO19_GRAD_CLIP),
        denoiser_variant=str(ALGO19_DENOISER_VARIANT),
        coordinate_score_mode=str(ALGO19_COORDINATE_SCORE_MODE),
    )
    rows = []
    for log in project['logs']:
        rows.append({
            'test': 'algo19_optimizer_trace',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            't_value': float(t_value),
            'lambda0': float(lambda0),
            **log,
            'PASS': True,
            'status': 'INFO',
        })
    return rows


debug_rows = []
for case in GRAPH_CASES:
    if int(case.graph_id) not in set(PPR_DEBUG_GRAPH_IDS):
        continue
    for t_value in PPR_DEBUG_T_VALUES:
        for lambda0 in PPR_DEBUG_LAMBDA0_VALUES:
            for M in PPR_DEBUG_M_VALUES:
                try:
                    debug_rows.append(fixed_t_paired_debug_row(case, t_value=float(t_value), lambda0=float(lambda0), M=int(M)))
                except Exception as exc:
                    debug_rows.append(error_row(
                        'algo19_paired_fixed_t_debug',
                        case.graph_id,
                        exc,
                        'paired_fixed_t_debug',
                        space_group=case.requested_sg,
                        t_value=float(t_value),
                        lambda0=float(lambda0),
                        M=int(M),
                    ))

paired_debug_df = dataframe_result('algo19_paired_fixed_t_debug', debug_rows)
paired_debug_df = paired_debug_df.sort_values(['graph', 't_value', 'lambda0', 'M']).reset_index(drop=True)
display(paired_debug_df)

trace_rows = []
for case in GRAPH_CASES:
    if int(case.graph_id) not in set(PPR_DEBUG_GRAPH_IDS):
        continue
    trace_rows.extend(optimizer_trace_rows(case, t_value=0.5, lambda0=PPR_DEFAULT_LAMBDA0, proj_steps=PPR_DEFAULT_PROJ_STEPS))

optimizer_trace_df = dataframe_result('algo19_optimizer_trace', trace_rows)
optimizer_trace_df = optimizer_trace_df.sort_values(['graph', 't_value', 'step']).reset_index(drop=True) if not optimizer_trace_df.empty else optimizer_trace_df
display(optimizer_trace_df)


KeyboardInterrupt: 

## Notes

- The main tuning metric is now `c_ppr_redenoise < c_renoise_redenoise`, not just whether the clean projected anchor looks excellent.
- The fixed-t debug table uses paired KLDM renoise noise to make the baseline-vs-PPR comparison fair.
- The sampler comparison keeps `M = 1` and projection every 50 steps by default, while the debug sweep explores the larger `M` and `lambda0` grid.
